### Import Required Dependencies

In [ ]:
# --- Imports ---
import pandas as pd
import numpy as np
import warnings
import logging
import matplotlib.pyplot as plt
import seaborn as sns
import re
import sys
from functools import reduce



from pathlib import Path
from functools import reduce
from sklearn.preprocessing import MinMaxScaler
from collections import Counter

# --- Setup ---
sns.set(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
warnings.filterwarnings('ignore')

# --- Paths ---
data_dir = Path("data")
output_dir = Path("outputs")
output_dir.mkdir(exist_ok=True)

log_dir = Path("logs")
log_dir.mkdir(exist_ok=True)

image_dir = Path("outputs/images")
image_dir.mkdir(parents=True, exist_ok=True)

# Clear existing handlers
logging.getLogger().handlers.clear()

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[
        logging.FileHandler(log_dir / "project.log", mode='w', encoding='utf-8'),
        logging.StreamHandler(sys.stdout)
    ]
)


print("Environment ready. Paths and logging configured.")



# Section 2 Load and Process Dataset (national and Arkansas)

## Section 2A: Load utility functions

In [ ]:
# Section 2A: Load utility functions and national datasets ----

# Import custom utility functions
from utils import (
    standardize_column_names,
    extract_key_indicators,
    filter_to_county_level,
    log_duplicate_attributes,
    clean_and_extract_year,
    nca_counties
)

data_dir = Path("data")
complete_dir = data_dir / "complete_sets"

complete_files = {
    "edu": "Education2023.csv",
    "pop": "PopulationEstimates.csv",
    "poverty": "Poverty2023.csv",
    "unemp": "Unemployment2023.csv"
}

# Load, clean, and extract year from each dataset
complete_data = {}

for key, filename in complete_files.items():
    path = complete_dir / filename
    try:
        df = pd.read_csv(path, encoding='cp1252')
        df = standardize_column_names(df)
        logging.info(f" {key} columns after cleaning: {df.columns.tolist()}")
        df = filter_to_county_level(df)
        df = clean_and_extract_year(df)

        # Rename fields if necessary
        df.rename(columns={'area_name': 'county'}, inplace=True)

        log_duplicate_attributes(df, key)
        complete_data[key] = df
        logging.info(f"Loaded {filename}: {df.shape[0]} rows")

    except Exception as e:
        logging.error(f" Failed to load {filename}: {e}")


# Confirmation message
logging.info("Utility functions from utils.py loaded successfully.")


## Section 2B: Load and process national datasets

In [ ]:
# Section 2B: Load and process national datasets

complete_dir = data_dir / "complete_sets"
complete_files = {
    "edu": "Education2023.csv",
    "pop": "PopulationEstimates.csv",
    "poverty": "Poverty2023.csv",
    "unemp": "Unemployment2023.csv"
}

state_lookup = None

# Choose your analysis year here
selected_year = '2023'

for key, filename in complete_files.items():
    path = complete_dir / filename
    try:
        # Load and standardize
        df = pd.read_csv(path, encoding='cp1252')
        df = standardize_column_names(df)
        df = filter_to_county_level(df)
        df.rename(columns={'area_name': 'county', 'fips_code': 'fips', 'fipstxt': 'fips'}, inplace=True)
        df['county'] = df['county'].str.strip().str.lower()

        # Extract year from attribute
        if 'attribute' in df.columns:
            df = clean_and_extract_year(df)

            # Only filter datasets where year tagging applies
            if key in ['pop', 'poverty', 'unemp']:
                df = df[df['attribute_year'] == selected_year]

        # Save state info once from education
        if key == 'edu':
            state_lookup = df[['county', 'state']].drop_duplicates().copy()
            state_lookup['county'] = state_lookup['county'].str.lower().str.strip()
            state_lookup['state'] = state_lookup['state'].str.upper().str.strip()

        log_duplicate_attributes(df, key)
        complete_data[key] = df
        logging.info(f" Loaded and processed {filename} with year filter: {selected_year if key != 'edu' else 'N/A'}")

    except Exception as e:
        logging.error(f" Failed to load {filename}: {e}")


### Section 2B.1 Determine most common year across all datasets

In [ ]:
## Section 2B.1: Discover Common Year Across Datasets

def get_attribute_year_counts(df):
    """Counts how often each extracted attribute_year appears."""
    if 'attribute' in df.columns:
        df = clean_and_extract_year(df)
        return Counter(df['attribute_year'].dropna().astype(str))
    return {}

year_summary = {}

# Loop through each dataset and extract year counts
for key, df in complete_data.items():
    year_counts = get_attribute_year_counts(df)
    year_summary[key] = year_counts

# Display results
print("Year coverage by dataset:")
all_years = set()

for key, counts in year_summary.items():
    print(f"\n {key.upper()}:")
    if counts:
        for year, count in sorted(counts.items()):
            print(f"  {year}: {count}")
            all_years.add(year)
    else:
        print("  No attribute_year values found.")

# Find common years across all datasets that have year values
datasets_with_years = [set(c.keys()) for c in year_summary.values() if c]
common_years = set.intersection(*datasets_with_years) if datasets_with_years else set()



print("\n Common years across all datasets with usable attribute_year:", sorted(common_years))


## Section 2C: Subset Arkansas and NCA Counties

In [ ]:
# Section 2C: Subset Arkansas and NCA Counties (No Full Merge)

# Define NCA counties (lowercase, cleaned)
nca_cleaned = [c.lower().strip() for c in nca_counties]

# Step 1: Function to filter, clean, and subset each dataset
def get_nca_subset(df, dataset_name):
    df = df.copy()
    df = df[df['state'].str.upper() == 'AR']
    df['county'] = (
        df['county']
        .str.replace(" county", "", regex=False)
        .str.replace(", ar", "", regex=False)
        .str.strip()
        .str.lower()
    )
    df_nca = df[df['county'].isin(nca_cleaned)].copy()
    logging.info(f"{dataset_name.upper()} → NCA rows: {df_nca.shape[0]}")
    return df_nca

# Step 2: Get cleaned NCA subsets
edu_nca = get_nca_subset(complete_data['edu'], 'edu')
poverty_nca = get_nca_subset(complete_data['poverty'], 'poverty')
unemp_nca = get_nca_subset(complete_data['unemp'], 'unemp')
pop_nca = get_nca_subset(complete_data['pop'], 'pop')

# Step 3: Pivot each to wide format
edu_wide = edu_nca.pivot_table(index='county', columns='attribute', values='value', aggfunc='first')
poverty_wide = poverty_nca.pivot_table(index='county', columns='attribute', values='value', aggfunc='first')
unemp_wide = unemp_nca.pivot_table(index='county', columns='attribute', values='value', aggfunc='first')
pop_wide = pop_nca.pivot_table(index='county', columns='attribute', values='value', aggfunc='first')

# Step 4: Merge NCA-wide datasets
df_nca = reduce(
    lambda left, right: pd.merge(left, right, on='county', how='outer'),
    [edu_wide, poverty_wide, unemp_wide, pop_wide]
)

# Step 5: Save and log
df_nca.to_csv(output_dir / f"nca_dataset_cleaned_{selected_year}.csv")
logging.info(f"NCA final dataset saved: nca_dataset_cleaned_{selected_year}.csv")
print(f"NCA dataset ready with shape: {df_nca.shape}")


## Section 3: Exploratory Data Analysis (EDA)

In [ ]:
# SECTION 3: Exploratory Data Analysis (EDA)

# Use a working copy of the NCA subset
df = df_nca.copy()
logging.info(f" Starting EDA on NCA dataset: {df.shape[0]} counties, {df.shape[1]} features")


### Section 3.1: Inspect Dataset

In [ ]:
# Full dataset overview
print("🔧 Info:")
display(df.info())

print("\n Descriptive Stats:")
display(df.describe(include='all'))

print("\n Sample Data:")
display(df.head())


### Section 3.2 Extract and Rename Key Variables

In [ ]:
## Section 3.2: Extract and Name Key Indicator Variables

#  Print all column names for inspection
print("\n All column names in df_nca:")
for col in df_nca.columns:
    print(col)

# 🔍 Extract key indicators
df_nca, used_columns, year = extract_key_indicators(df_nca, min_year=2022)

# Print matched column summary
print("Most common year in column names:", year)
print("Education columns used:", used_columns['education'])
print("Poverty columns used:", used_columns['poverty'])
print("Unemployment columns used:", used_columns['unemployment'])
print("Population columns used:", used_columns['population'])

# Compute education percentages (relative to population)
df_nca['BachelorsDegreePct'] = (df_nca['BachelorsDegreeRate'] / df_nca['Population']) * 100
df_nca['HighSchoolGradPct'] = (df_nca['HighSchoolGradRate'] / df_nca['Population']) * 100

# Define clean variable set for EDA (use percentages)
variables = ['BachelorsDegreePct', 'HighSchoolGradPct', 'PovertyRate', 'UnemploymentRate', 'Population']

# Preview the cleaned dataset
print("\n Preview of standardized indicators:")
display(df_nca[variables].head())

# Debug: Print all matched and available columns
print("\n Matched columns by indicator:")
for key, cols in used_columns.items():
    print(f"  - {key}: {cols}")

print("\nAll available columns:")
print(df_nca.columns.tolist())



### Section 3.3 Visualize Education Levels

In [ ]:
education_cols = used_columns['education'][:2]  # Bachelor's + High School

# Education distribution across counties
title = "Education Indicators by County"
df[education_cols].T.plot(kind='bar', figsize=(14, 6), title=title)
plt.ylabel("Percent or Count")
plt.xticks(rotation=45, ha='right')
plt.tight_layout()

# Standardize filename
filename = title.lower().replace(" ", "_").replace("’", "").replace("'", "")
plt.savefig(image_dir / f"{filename}.png", dpi=300, bbox_inches='tight')
logging.info(f" Saved: {filename}.png")

plt.show()



### Section 3.4: Distribution Plots (Histograms & KDE)

In [ ]:
variables = ['PovertyRate', 'UnemploymentRate', 'HighSchoolGradPct', 'BachelorsDegreePct']

for var in variables:
    plt.figure(figsize=(8, 4))
    
    title = f"Distribution of {var}"
    
    # Use the correct DataFrame
    sns.histplot(df_nca[var], kde=True, bins=20)
    
    plt.title(title)
    plt.xlabel(var)
    plt.ylabel('Frequency')
    plt.tight_layout()
    
    # Save with safe filename
    filename = title.lower().replace(" ", "_").replace("’", "").replace("'", "")
    plt.savefig(image_dir / f"{filename}.png", dpi=300, bbox_inches='tight')
    logging.info(f"📷 Saved: {filename}.png")
    
    plt.show()




### Section 3.5: Correlation Heatmap

In [ ]:
# Correlation matrix and heatmap
corr_vars = df_nca[['PovertyRate', 'UnemploymentRate', 'HighSchoolGradRate', 'BachelorsDegreeRate', 'Population']]
corr_matrix = corr_vars.corr()

# Define title
title = "Correlation Between Key Indicators"

# Plot
plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".2f")
plt.title(title)
plt.tight_layout()

# Generate safe filename from title
filename = title.lower().replace(" ", "_").replace("’", "").replace("'", "")
plt.savefig(image_dir / f"{filename}.png", dpi=300, bbox_inches='tight')
logging.info(f"📷 Saved: {filename}.png")

plt.show()


### Section 3.6: Key Relationships (Scatter Plots)

In [ ]:
# Scatter Plots of Key Relationships
# 1. Bachelor's Degree vs Poverty
title = "Bachelor's Degree Rate vs. Poverty Rate"
sns.scatterplot(x='BachelorsDegreeRate', y='PovertyRate', data=df_nca)
plt.title(title)
plt.xlabel("Bachelor's Degree (%)")
plt.ylabel("Poverty Rate (%)")
plt.tight_layout()
filename = title.lower().replace(" ", "_").replace("’", "").replace("'", "")
plt.savefig(image_dir / f"{filename}.png", dpi=300, bbox_inches='tight')
logging.info(f"📷 Saved: {filename}.png")
plt.show()

# 2. High School Grad Rate vs Unemployment
title = "High School Grad Rate vs. Unemployment Rate"
sns.scatterplot(x='HighSchoolGradRate', y='UnemploymentRate', data=df_nca)
plt.title(title)
plt.xlabel("High School Grad (%)")
plt.ylabel("Unemployment Rate (%)")
plt.tight_layout()
filename = title.lower().replace(" ", "_").replace("’", "").replace("'", "")
plt.savefig(image_dir / f"{filename}.png", dpi=300, bbox_inches='tight')
logging.info(f"📷 Saved: {filename}.png")
plt.show()

# 3. Population vs Poverty Rate
title = "Population vs. Poverty Rate"
sns.scatterplot(x='Population', y='PovertyRate', data=df_nca)
plt.title(title)
plt.xlabel("Population")
plt.ylabel("Poverty Rate (%)")
plt.tight_layout()
filename = title.lower().replace(" ", "_").replace("’", "").replace("'", "")
plt.savefig(image_dir / f"{filename}.png", dpi=300, bbox_inches='tight')
logging.info(f"📷 Saved: {filename}.png")
plt.show()


### Section 3.7: Outlier Detection with BoxPlots

In [ ]:
# --- Boxplots for Outlier Detection (with title-based save) ---
for var in variables:
    plt.figure(figsize=(8, 4))

    # Define title and filename
    title = f"Boxplot of {var}"
    sns.boxplot(x=df_nca[var])
    plt.title(title)
    plt.tight_layout()

    # Standardize filename
    filename = title.lower().replace(" ", "_").replace("’", "").replace("'", "")
    plt.savefig(image_dir / f"{filename}.png", dpi=300, bbox_inches='tight')
    logging.info(f"📷 Saved: {filename}.png")

    plt.show()


### Section 3.8 ParPlot for Key Indicators

In [ ]:
# Pairplot for Key Indicators
sns.pairplot(df_nca[variables], diag_kind='kde')
plt.suptitle("Pairwise Relationships Between Key Indicators", y=1.02)
plt.tight_layout()
plt.savefig(image_dir / "pairplot_key_indicators.png", dpi=300, bbox_inches='tight')
logging.info("📷 Saved: pairplot_key_indicators.png")
plt.show()


### Step 4: Visualize Distributions (Histograms & KDE)

### Step 5: Correlation Matrix and Heatmap

### Step 6: Scatter Plots for Key Relationships

### Step 7: Identify Outlier Counties with Boxplots

### Step 8: Log-Transform Population (Optional)
If the population is skewed: